# Feature Weighting

Runs a 7-model gene weighting pipeline on synthetic data first, to check it works before using real data.

**Real data guard:** `USE_REAL_DATA` must stay `False` in this AI session. Only set `True` for your own local, offline run (see `CLAUDE.md`).

**Limitations:**
- Only 8 samples (4 SHAM, 4 C26) — the paper used 88. Expect noisy, oversized gene lists.
- Male and female are kept separate, not pooled.
- Stops at picking genes — no classifier training yet.


In [ ]:
USE_REAL_DATA = True # keep False here. Only set True in your own local run, never inside an AI tool session, per CLAUDE.md.

import time
import numpy as np
import pandas as pd

from attribute_weighting import (
    weight_by_information_gain,
    weight_by_chi_squared,
    weight_by_gini_index,
    weight_by_rule,
    weight_by_information_gain_ratio,
    weight_by_symmetric_uncertainty,
    weight_by_relief,
    run_attribute_weighting,
)
from lasso_selection import cpm_log_normalize

SEXES = ["female", "male"]
EXPECTED_GENE_COUNTS = {"female": 2881, "male": 4257}
DATA_DIR = "restructured"


## Step 1. Load data and check shapes

In [ ]:
data = {}

for sex in SEXES:
    if USE_REAL_DATA:
        df = pd.read_csv(f"{DATA_DIR}/{sex}_expr.csv")
        with open(f"{DATA_DIR}/{sex}_filtered_gene_symbols.txt") as f:
            gene_cols = [line.strip() for line in f if line.strip()]
    else:
        df = pd.read_csv(f"{DATA_DIR}/{sex}_expr_synthetic.csv")
        gene_cols = [c for c in df.columns if c not in ("Sample", "Group", "Label")]

    assert df.shape[0] == 8, f"{sex}: expected 8 samples, got {df.shape[0]}"
    assert len(gene_cols) == EXPECTED_GENE_COUNTS[sex], (
        f"{sex}: expected {EXPECTED_GENE_COUNTS[sex]} genes, got {len(gene_cols)}"
    )
    assert (df["Group"].value_counts() == 4).all(), f"{sex}: expected a 4/4 Group split"
    assert (df["Label"] == (df["Group"] == "C26").astype(int)).all(), (
        f"{sex}: Label does not match Group for some samples"
    )

    # CPM + log2 normalize before any feature selection — raw counts carry each
    # sample's sequencing depth as a confound (our own library sizes vary by
    # ~20% sample to sample), which a per-gene z-score inside LOOCV does not
    # correct for. Found this changes which genes come out on top, not just a
    # formality.
    X_normalized = cpm_log_normalize(df[gene_cols])

    data[sex] = {"X": X_normalized, "y": df["Label"]}
    print(f"{sex}: OK, {df.shape[0]} samples x {len(gene_cols)} genes, 4/4 Group split, Label matches Group, CPM+log2 normalized")


## Step 2. Sanity check on toy data

One gene with an obvious signal, four pure noise genes. Every model should rank the signal gene highest — confirms the 7 models work before running on real genes.

In [ ]:
toy_y = pd.Series([0, 0, 0, 0, 1, 1, 1, 1])

rng = np.random.default_rng(42)
toy_X = pd.DataFrame({
    "Signal": [5, 8, 10, 12, 120, 125, 130, 135],  # clean SHAM-low / C26-high split
    "Noise1": rng.uniform(0, 150, size=8),
    "Noise2": rng.uniform(0, 150, size=8),
    "Noise3": rng.uniform(0, 150, size=8),
    "Noise4": rng.uniform(0, 150, size=8),
})
noise_cols = ["Noise1", "Noise2", "Noise3", "Noise4"]

toy_models = {
    "IG": weight_by_information_gain,
    "Chi2": weight_by_chi_squared,
    "Gini": weight_by_gini_index,
    "Rule": weight_by_rule,
    "IGRatio": weight_by_information_gain_ratio,
    "Uncertainty": weight_by_symmetric_uncertainty,
    "Relief": weight_by_relief,
}

for name, fn in toy_models.items():
    scores = fn(toy_X, toy_y)
    worst_noise = scores[noise_cols].max()
    passed = scores["Signal"] > worst_noise
    status = "PASS" if passed else "FAIL"
    print(f"{status}  {name:12s}  Signal={scores['Signal']:.4f}  best noise={worst_noise:.4f}")
    assert passed, f"{name} did not score the signal gene above the noise genes"


## Step 3. Run all 7 models

Combines the 7 model scores into one `WeightSum` (0–7) per gene. `Selected` = `WeightSum > 5`.

In [ ]:
results = {}

for sex in SEXES:
    t0 = time.time()
    result = run_attribute_weighting(data[sex]["X"], data[sex]["y"])
    elapsed = time.time() - t0

    numeric_cols = ["IG", "Chi2", "Gini", "Rule", "IGRatio", "Uncertainty", "Relief", "WeightSum", "Rank"]
    n_nan = result[numeric_cols].isna().sum().sum()
    n_inf = np.isinf(result[numeric_cols].to_numpy()).sum()
    assert n_nan == 0, f"{sex}: found {n_nan} NaN values in the weight table"
    assert n_inf == 0, f"{sex}: found {n_inf} Inf values in the weight table"

    results[sex] = result
    n_selected = result["Selected"].sum()
    n_total = result.shape[0]
    print(f"{sex}: {elapsed:.1f}s, {n_total} genes, {n_selected} selected ({n_selected/n_total:.1%}), no NaN/Inf")


## Step 3.5. Stability check (leave-one-pair-out)

`WeightSum > 5` is too lenient at n=8 — it keeps most genes. This reruns the pipeline on 16 resampled folds (each drops one SHAM + one C26 sample) and ranks genes by how often they land in the top 25/50. Takes a few minutes per sex.

This is a soft, draft ranking, not a proven gene list — treat it as candidates to check against known biology.

In [ ]:
from attribute_weighting import run_stability_selection

stability_results = {}

for sex in SEXES:
    t0 = time.time()
    stability = run_stability_selection(data[sex]["X"], data[sex]["y"], top_k=(25, 50))
    elapsed = time.time() - t0

    stability_results[sex] = stability
    combined = results[sex].join(stability)
    results[sex] = combined

    n_stable_top25 = (stability["Top25Freq"] > 0).sum()
    print(f"{sex}: {elapsed:.1f}s for 16 folds")
    print(f"  genes ever in top 25 across folds: {n_stable_top25}")
    print(f"  top 10 by Top25Freq / MeanRank:")
    print(combined.sort_values(["Top25Freq", "MeanRank"], ascending=[False, True])
          [["WeightSum", "Rank", "Top25Freq", "Top50Freq", "MeanRank"]].head(10))


## Step 3.6. Three sorted views of the top 50

Same combined table, three different sort orders, top 50 rows each: by
`Top25Freq` (stability-first), by `MeanRank` (average rank across the 16
folds), and by `WeightSum` (the single full-data run only, no resampling —
shown for comparison, not as a recommendation).

In [ ]:
pd.set_option("display.max_rows", 50)
view_cols = ["WeightSum", "Rank", "Top25Freq", "Top50Freq", "MeanRank"]

for sex in SEXES:
    print(f"=== {sex}: top 50 by Top25Freq (then MeanRank) ===")
    print(results[sex].sort_values(["Top25Freq", "MeanRank"], ascending=[False, True])[view_cols].head(50))
    print()


In [ ]:
for sex in SEXES:
    print(f"=== {sex}: top 50 by MeanRank ===")
    print(results[sex].sort_values("MeanRank", ascending=True)[view_cols].head(50))
    print()


In [ ]:
for sex in SEXES:
    print(f"=== {sex}: top 50 by WeightSum ===")
    print(results[sex].sort_values("WeightSum", ascending=False)[view_cols].head(50))
    print()


## Step 3.7. Check specific genes of interest

Look up where the classic myostatin-myogenin/UPS axis genes land, regardless
of whether they made any top-N cutoff above.

In [ ]:
genes_of_interest = ["Myog", "Fbxo32", "Trim63"]

for sex in SEXES:
    print(f"=== {sex} ===")
    lookup = results[sex].reindex(genes_of_interest)[view_cols]
    lookup["Rank"] = lookup["Rank"].astype("Int64")
    lookup["TotalGenes"] = results[sex].shape[0]
    print(lookup)
    print()


## Step 3.8. Correlation checks between top hits and hypothesis genes

Pearson and Spearman shown side by side — at n=8, a single sample can swing
Pearson a lot, so Spearman (rank-based) is a useful cross-check. Treat both as
very noisy point estimates, not conclusions.

In [ ]:
from scipy.stats import pearsonr, spearmanr

def print_corr(sex, gene_a, gene_b):
    X = data[sex]["X"]
    if gene_a not in X.columns or gene_b not in X.columns:
        missing = gene_a if gene_a not in X.columns else gene_b
        print(f"  {gene_a} vs {gene_b}: '{missing}' not in {sex} gene set")
        return
    r, p_pearson = pearsonr(X[gene_a], X[gene_b])
    rho, p_spearman = spearmanr(X[gene_a], X[gene_b])
    print(f"  {gene_a} vs {gene_b}:  Pearson r={r:+.3f} (p={p_pearson:.3f})   Spearman rho={rho:+.3f} (p={p_spearman:.3f})")

print("=== female: Per3 vs Foxo1 / Foxo3 / Myog / Trim63 ===")
for partner in ["Foxo1", "Foxo3", "Myog", "Trim63"]:
    print_corr("female", "Per3", partner)


In [ ]:
print("=== female: Dbp vs Ky ===")
print_corr("female", "Dbp", "Ky")

print("=== female: Dbp vs Foxo1 / Foxo3 / Myog / Trim63 ===")
for partner in ["Foxo1", "Foxo3", "Myog", "Trim63"]:
    print_corr("female", "Dbp", partner)

print("=== female: Ky vs Foxo1 / Foxo3 / Myog / Trim63 ===")
for partner in ["Foxo1", "Foxo3", "Myog", "Trim63"]:
    print_corr("female", "Ky", partner)


In [ ]:
print("=== male: Nrep vs Foxo1 / Foxo3 / Myog / Trim63 ===")
for partner in ["Foxo1", "Foxo3", "Myog", "Trim63"]:
    print_corr("male", "Nrep", partner)

print("=== male: Ppp1r3c vs Foxo1 / Foxo3 / Myog / Trim63 ===")
for partner in ["Foxo1", "Foxo3", "Myog", "Trim63"]:
    print_corr("male", "Ppp1r3c", partner)


## Step 4. Check injected genes (synthetic only)

Synthetic data has ~25 genes with an artificial 4x signal boost. This checks the pipeline ranks them higher than the rest. Skipped automatically on real data.

In [ ]:
if USE_REAL_DATA:
    print("Skipping Step 4 — no injected ground truth on real data.")
else:
    for sex in SEXES:
        with open(f"{DATA_DIR}/{sex}_synthetic_injected_genes.txt") as f:
            injected_genes = [line.strip() for line in f if line.strip()]

        result = results[sex]
        injected_mask = result.index.isin(injected_genes)

        frac_selected = result.loc[injected_mask, "Selected"].mean()
        frac_top50 = (result.loc[injected_mask, "Rank"] <= 50).mean()
        mean_ws_injected = result.loc[injected_mask, "WeightSum"].mean()
        mean_ws_other = result.loc[~injected_mask, "WeightSum"].mean()

        print(f"--- {sex} ---")
        print(f"  injected genes selected:      {frac_selected:.0%}")
        print(f"  injected genes in top 50:     {frac_top50:.0%}")
        print(f"  mean WeightSum, injected:     {mean_ws_injected:.2f}")
        print(f"  mean WeightSum, other genes:  {mean_ws_other:.2f}")

        assert mean_ws_injected > mean_ws_other, (
            f"{sex}: injected genes should score higher on average than the rest — pipeline may be broken"
        )

## Step 5. Check male and female are independent

Quick check that male and female results aren't accidentally duplicated.

In [ ]:
top50 = {sex: set(results[sex].index[results[sex]["Rank"] <= 50]) for sex in SEXES}
overlap = len(top50["female"] & top50["male"]) / 50

print(f"female/male top-50 gene overlap: {overlap:.0%}")
assert overlap < 0.9, "female and male top-50 gene sets are suspiciously similar — check for accidental duplication"
print("OK — male and female results are independent")


## Step 6. Save results

Saves the full weight table, the `WeightSum > 5` gene list, and a stricter 30-gene draft shortlist from the Step 3.5 stability ranking.

In [ ]:
COLUMN_ORDER = [
    "IG", "IGRatio", "Rule", "Chi2", "Gini", "Uncertainty", "Relief", "WeightSum", "Rank", "Selected",
    "StabilityScore", "MeanWeightSum", "MeanRank", "Top25Freq", "Top50Freq",
]
SHORTLIST_SIZE = 30

for sex in SEXES:
    result = results[sex][COLUMN_ORDER].copy()
    result.index.name = "Gene"
    result = result.reset_index()

    weights_path = f"{DATA_DIR}/{sex}_attribute_weights.csv"
    genes_path = f"{DATA_DIR}/{sex}_signature_genes.txt"
    shortlist_path = f"{DATA_DIR}/{sex}_signature_genes_draft_shortlist.txt"

    result.to_csv(weights_path, index=False)
    signature_genes = result.loc[result["Selected"], "Gene"].tolist()
    with open(genes_path, "w") as f:
        f.write("\n".join(signature_genes))

    shortlist = (
        result.sort_values(["Top25Freq", "Top50Freq", "MeanRank"], ascending=[False, False, True])
        .head(SHORTLIST_SIZE)["Gene"]
        .tolist()
    )
    with open(shortlist_path, "w") as f:
        f.write("\n".join(shortlist))

    print(f"{sex}: saved {result.shape[0]}-gene weight table -> {weights_path}")
    print(f"{sex}: saved {len(signature_genes)} signature genes (WeightSum > 5) -> {genes_path}")
    print(f"{sex}: saved {len(shortlist)}-gene draft shortlist (stability-ranked) -> {shortlist_path}")
